# 15 — Product Comparison Evaluation

Benchmark ثابت Product Comparison:

- 15 سناریو: 5 DEV + 10 TEST
- دو و سه محصولی
- metadata-deterministic، experiential، conflict، negative constraint و insufficient-evidence
- LLM judge برای کیفیت پاسخ
- checks قطعی برای citation ownership، coverage و winnerهای متادیتایی
- checkpoint/resume برای جلوگیری از تکرار API call

**نکته:** امتیازهای judge یک `LLM-assisted proxy` هستند، نه human gold مستقل.

In [1]:
import json
import os
from pathlib import Path
import sys

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.rag.config import load_config
from src.rag.evaluation.comparison_dataset import (
    ComparisonEvaluationDataset,
)
from src.rag.evaluation.comparison_evaluator import (
    ProductComparisonEvaluator,
    build_comparison_report,
    comparison_failure_summary,
    export_manual_review_csv,
)
from src.rag.evaluation.comparison_metrics import (
    summarize_comparison_results,
)
from src.rag.evaluation.comparison_runtime import (
    load_comparison_evaluation_context,
)

In [2]:
eval_config = load_config(
    PROJECT_ROOT
    / "configs"
    / "comparison_evaluation.yaml"
)["comparison_evaluation"]

dataset = ComparisonEvaluationDataset.load(
    PROJECT_ROOT
    / "configs"
    / "comparison_eval_cases.yaml"
)

cases = dataset.to_frame()

display(
    cases.groupby(
        [
            "split",
            "case_type",
        ]
    ).size().rename("cases")
)

print("Total cases:", len(cases))
print("Comparison calls:", len(cases))
print("Judge calls:", len(cases))

split  case_type             
dev    experiential              2
       experiential_conflict     1
       metadata_deterministic    2
test   experiential              2
       experiential_negative     3
       insufficient_evidence     1
       metadata_deterministic    2
       three_way_experiential    2
Name: cases, dtype: int64

Total cases: 15
Comparison calls: 15
Judge calls: 15


In [3]:
context = load_comparison_evaluation_context(
    project_root=PROJECT_ROOT,
    api_key=os.environ["METIS_API_KEY"],
    base_url=os.environ["METIS_BASE_URL"],
)

missing = dataset.validate_products(
    context.product_documents
)

if missing:
    display(pd.DataFrame(missing))
    raise ValueError(
        "Some benchmark product IDs are missing. "
        "Fix the case file before spending API calls."
    )

print(
    "All benchmark product IDs exist in products_search.parquet."
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

All benchmark product IDs exist in products_search.parquet.


In [4]:
EVAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "comparison"
)

EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

evaluator = ProductComparisonEvaluator(
    comparison_pipeline=(
        context.comparison
    ),
    judge=context.judge,
    dataset=dataset,
    weights=eval_config["weights"],
)

results = evaluator.evaluate(
    checkpoint_path=(
        EVAL_DIR
        / "comparison_checkpoint.jsonl"
    ),
    resume=True,
)

display(
    results[
        [
            "case_id",
            "split",
            "case_type",
            "status",
        ]
    ]
)

print(
    "Successful:",
    int((results["status"] == "ok").sum()),
    "/",
    len(results),
)

,case_id,split,case_type,status
0,c001,dev,experiential_conflict,ok
1,c002,dev,metadata_deterministic,ok
2,c003,dev,experiential,ok
3,c004,dev,metadata_deterministic,ok
4,c005,dev,experiential,ok
5,c006,test,experiential,ok
6,c007,test,experiential,ok
7,c008,test,experiential_negative,ok
8,c009,test,experiential_negative,ok
9,c010,test,experiential_negative,ok


Successful: 15 / 15


In [5]:
ok = results[
    results["status"] == "ok"
].copy()

metric_columns = [
    "overall_score",
    "correctness",
    "groundedness",
    "criterion_coverage",
    "conflict_handling",
    "recommendation_calibration",
    "relevance",
    "instruction_following",
    "safety",
    "citation_validity",
    "citation_ownership_rate",
    "assessment_product_coverage",
    "deterministic_winner_accuracy",
    "no_winner_accuracy",
]

display(
    ok.groupby("split")[
        metric_columns
    ].mean(numeric_only=True).round(3)
)

print("TEST only:")

display(
    ok[
        ok["split"] == "test"
    ][
        metric_columns
    ].mean(numeric_only=True).round(3)
    .rename("score")
    .to_frame()
)

,overall_score,correctness,groundedness,criterion_coverage,conflict_handling,recommendation_calibration,relevance,instruction_following,safety,citation_validity,citation_ownership_rate,assessment_product_coverage,deterministic_winner_accuracy,no_winner_accuracy
split,,,,,,,,,,,,,,
dev,4.97,5.0,5.0,5.0,5.0,4.8,5.0,5.0,5.0,1.0,1.0,1.0,1.0,NaN
test,4.88,4.8,4.8,5.0,5.0,4.8,4.9,5.0,5.0,1.0,1.0,1.0,1.0,1.0


TEST only:


,score
overall_score,4.88
correctness,4.80
groundedness,4.80
criterion_coverage,5.00
conflict_handling,5.00
recommendation_calibration,4.80
relevance,4.90
instruction_following,5.00
safety,5.00
citation_validity,1.00


In [6]:
summary = summarize_comparison_results(
    ok
)

print("By case type:")
display(
    summary["by_type"].round(3)
)

print("Failures:")
failures = comparison_failure_summary(
    ok
)
display(failures)

failure_cases = ok[
    ok["failure_tags"].map(
        lambda values: bool(values)
    )
][
    [
        "case_id",
        "split",
        "case_type",
        "query",
        "overall_score",
        "failure_tags",
        "judge_summary_reason",
    ]
]

display(failure_cases)

By case type:


,case_type,overall_score,correctness,groundedness,criterion_coverage,conflict_handling,recommendation_calibration,relevance,instruction_following,safety,citation_validity,citation_ownership_rate,assessment_product_coverage,deterministic_winner_accuracy,no_winner_accuracy
0,experiential,4.800,4.750,4.750,5.0,5.0,4.500,4.75,5.0,5.0,1.0,1.0,1.0,NaN,NaN
1,experiential_conflict,5.000,5.000,5.000,5.0,5.0,5.000,5.00,5.0,5.0,1.0,1.0,1.0,NaN,NaN
2,experiential_negative,4.817,4.667,4.667,5.0,5.0,4.667,5.00,5.0,5.0,1.0,1.0,1.0,NaN,NaN
3,insufficient_evidence,5.000,5.000,5.000,5.0,5.0,5.000,5.00,5.0,5.0,1.0,1.0,1.0,NaN,1.0
4,metadata_deterministic,5.000,5.000,5.000,5.0,5.0,5.000,5.00,5.0,5.0,1.0,1.0,1.0,1.0,NaN
5,three_way_experiential,5.000,5.000,5.000,5.0,5.0,5.000,5.00,5.0,5.0,1.0,1.0,1.0,NaN,NaN


Failures:


,failure_tag,count
0,contradicts_evidence,1
1,unsupported_claim,1
2,insufficient_evidence_mishandled,1


,case_id,split,case_type,query,overall_score,failure_tags,judge_summary_reason
5,c006,test,experiential,برای پوست چرب این دو ضد آفتاب را از نظر حس روی...,4.35,[contradicts_evidence],پاسخ در مجموع مقایسه‌ای دقیق و مبتنی بر شواهد ...
9,c010,test,experiential_negative,این دو ماشین اصلاح را از نظر نرمی اصلاح، سوزش ...,4.45,"[unsupported_claim, insufficient_evidence_mish...",پاسخ عمدتاً دقیق و مستند است و فیلیپس را بر پا...


In [7]:
telemetry_columns = [
    "comparison_total_latency_ms",
    "judge_latency_ms",
    "end_to_end_latency_ms",
    "comparison_total_tokens",
    "judge_total_tokens",
    "end_to_end_tokens",
]

telemetry_summary = pd.DataFrame(
    {
        "mean": ok[
            telemetry_columns
        ].mean(),
        "p95": ok[
            telemetry_columns
        ].quantile(0.95),
        "sum": ok[
            telemetry_columns
        ].sum(),
    }
)

display(
    telemetry_summary.round(2)
)

if (
    "end_to_end_cost_usd"
    in ok.columns
    and ok[
        "end_to_end_cost_usd"
    ].notna().any()
):
    print(
        "Estimated total evaluation cost USD:",
        round(
            float(
                ok[
                    "end_to_end_cost_usd"
                ].sum()
            ),
            4,
        ),
    )

,mean,p95,sum
comparison_total_latency_ms,9422.29,13223.25,141334.41
judge_latency_ms,8513.24,12831.36,127698.59
end_to_end_latency_ms,17939.94,25191.05,269099.17
comparison_total_tokens,1555.73,2166.60,23336.00
judge_total_tokens,2506.53,3328.30,37598.00
end_to_end_tokens,4062.27,5399.70,60934.00


In [8]:
deterministic = ok[
    ok[
        "deterministic_winner_available"
    ]
][
    [
        "case_id",
        "split",
        "query",
        "winner_rule",
        "deterministic_expected_winner",
        "overall_winner_product_id",
        "deterministic_winner_accuracy",
    ]
]

print("Deterministic metadata winner checks:")
display(deterministic)

insufficient = ok[
    ok["expect_no_winner"]
][
    [
        "case_id",
        "query",
        "overall_winner_product_id",
        "insufficient_evidence",
        "no_winner_accuracy",
        "recommendation_calibration",
        "failure_tags",
    ]
]

print("Insufficient-evidence stress test:")
display(insufficient)

Deterministic metadata winner checks:


,case_id,split,query,winner_rule,deterministic_expected_winner,overall_winner_product_id,deterministic_winner_accuracy
1,c002,dev,فقط بر اساس قیمت فعلی، کدام ضد آفتاب ارزان‌تر ...,lower_price,6283256.0,6283256.0,1.0
3,c004,dev,فقط با تکیه بر امتیاز ثبت‌شده در متادیتا بگو ک...,higher_rating,NaN,NaN,1.0
12,c013,test,فقط بر اساس قیمت فعلی، کدام ضد آفتاب ژیناژن ار...,lower_price,6283256.0,6283256.0,1.0
13,c014,test,فقط بر اساس امتیاز متادیتا بگو کدام ضد آفتاب ا...,higher_rating,6733478.0,6733478.0,1.0


Insufficient-evidence stress test:


,case_id,query,overall_winner_product_id,insufficient_evidence,no_winner_accuracy,recommendation_calibration,failure_tags
14,c015,کدام‌یک از این دو آبرسان تا عمق ۱۰ متر ضدآب اس...,NaN,True,1.0,5.0,[]


In [9]:
results.to_csv(
    EVAL_DIR
    / "comparison_results.csv",
    index=False,
)

flat_columns = [
    column
    for column in results.columns
    if column not in {
        "product_ids",
        "product_metadata",
        "criteria",
        "failure_tags",
    }
]

results[
    flat_columns
].to_parquet(
    EVAL_DIR
    / "comparison_metrics.parquet",
    index=False,
)

export_manual_review_csv(
    results,
    EVAL_DIR
    / "comparison_manual_review.csv",
)

report = build_comparison_report(
    results
)

with (
    EVAL_DIR
    / "comparison_report.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        report,
        handle,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

print(
    "Saved evaluation artifacts to:",
    EVAL_DIR,
)

report

Saved evaluation artifacts to: /home/ali/Desktop/projects/digikala-ai-assistant/data/evaluation/comparison


{'case_count': 15,
 'successful_cases': 15,
 'error_count': 0,
 'overall': {'overall_score': 4.91,
  'correctness': 4.866666666666666,
  'groundedness': 4.866666666666666,
  'criterion_coverage': 5.0,
  'conflict_handling': 5.0,
  'recommendation_calibration': 4.8,
  'relevance': 4.933333333333334,
  'instruction_following': 5.0,
  'safety': 5.0,
  'citation_validity': 1.0,
  'citation_ownership_rate': 1.0,
  'assessment_product_coverage': 1.0,
  'deterministic_winner_accuracy': 1.0,
  'no_winner_accuracy': 1.0},
 'telemetry': {'mean_comparison_total_latency_ms': 9422.29382993343,
  'p95_comparison_total_latency_ms': 13223.252050799963,
  'mean_judge_latency_ms': 8513.239475533434,
  'p95_judge_latency_ms': 12831.355206099575,
  'mean_end_to_end_latency_ms': 17939.944816866653,
  'p95_end_to_end_latency_ms': 25191.04827149931,
  'mean_comparison_total_tokens': 1555.7333333333333,
  'p95_comparison_total_tokens': 2166.5999999999995,
  'mean_judge_total_tokens': 2506.5333333333333,
  'p9